# End-to-end Ginsu application

_____________________________
This demo notebook is split in 2 parts:

- **Machine Learning modelling**

This part implements a basic classification pipeline on the [Titanic dataset](https://www.openml.org/search?type=data&sort=runs&id=40945&status=active) to predict if a passanger survived.

- **Model debugging with Ginsu**

This part identifies slices where the training error of the model is significantly higher, thanks to [sliceline](https://github.com/DataDome/sliceline).

## Machine Learning modelling

The pipeline is composed of 2 steps:
1. The preprocessor: to transform raw data into numerical data without NaN,
2. The classifier: a [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) with default parameters.

The training error is the element-wise [log loss](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.log_loss.html).

In [ ]:
# import useful modules
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

# fetch titanic dataset
X, y = fetch_openml("titanic", version=1, as_frame=True, return_X_y=True)
X.drop(
    ["cabin", "boat", "body", "home.dest", "name", "ticket"],
    axis=1,
    inplace=True,
)

# define pipeline
cat_cols = X.select_dtypes("category").columns
num_cols = X.select_dtypes("number").columns

cat_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ),
    ]
)

num_transformer = Pipeline(steps=[("imputer", KNNImputer(n_neighbors=5))])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_cols),
        ("cat", cat_transformer, cat_cols),
    ]
)

clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

# training
clf.fit(X, y)

# predict on training data
y_proba = clf.predict_proba(X)[:, 1]  # score of being a '1'

# compute element-wise log loss (the lower, the better)
eps = 1e-15
y_proba = np.clip(y_proba, eps, 1 - eps)
y = y.astype(int)

training_errors = -(y * np.log(y_proba) + (1 - y) * np.log(1 - y_proba))

## Model debbuging with Ginsu

**Ginsu considers all the columns of the input dataset as categorical.**

So, to get more relevant slices, the following numerical features should be discretized:
- `age`
- `fare`

Indeed, those columns as-is would lead to poor exploitable results. We would rather have range of values to specific value in our slices definition.

To discretize them and compute their bins, we use [OptBinning](http://gnpalencia.org/optbinning/) but feel free to experiment other binning implementations.

Ginsu configuration:
- `alpha = 0.95`: we are interested in small slice with high log loss.
- `k = 1`: we want Ginsu to find the rules with the best score.
- `max_l = X_trans.shape[1]`: we want Ginsu to be able to use all of the inputs columns.
- `min_sup = 1`: because the input dataset is relatively small, we do not add constraint regarding the minimal support.

In [ ]:
# import Ginsu and binning class
from ginsu import Slicefinder
from optbinning import ContinuousOptimalBinning

# dataset before prediction
X_trans = pd.DataFrame(
    clf[0].transform(X), columns=clf[0].get_feature_names_out()
)

# `age` and `fare` have to be bined
columns_to_bin = ["num__age", "num__fare"]

optimal_binner = ContinuousOptimalBinning()

X_trans[columns_to_bin] = np.array(
    [
        optimal_binner.fit_transform(
            X_trans[col], training_errors, metric="bins"
        )
        for col in columns_to_bin
    ]
).T

# fitting Ginsu
sf = Slicefinder(
    alpha=0.95, k=1, max_l=X_trans.shape[1], min_sup=1, verbose=True
)

sf.fit(X_trans, training_errors)

In [ ]:
# slices found
pd.DataFrame(
    sf.top_slices_,
    columns=sf.feature_names_in_,
    index=sf.get_feature_names_out(),
)

**Note:**

We found 40 slices with `k` set to 1. As described in the documentation, it means that those 40 slices have the same score.

**In fact, they target the same subset of data.**

_(`None` values refer to unused features in each slices.)_

In [ ]:
from sklearn.metrics import log_loss

# select one slice
slice_index = 0
current_slice = sf.top_slices_[slice_index]

# create a pandas filter
predicate_conditions = [
    X_trans[feature_name] == feature_value
    for feature_name, feature_value in zip(sf.feature_names_in_, current_slice)
    if feature_value is not None
]
condition = " & ".join(
    [f"@predicate_conditions[{i}]" for i in range(len(predicate_conditions))]
)

# get slice element indices
indices = X_trans.query(condition).index

print("Model log loss on:")
print(f"- the full dataset ({X.shape[0]} passengers):", log_loss(y, y_proba))
print(
    f"- the selected slice ({len(indices)} passengers):",
    log_loss(y.iloc[indices], y_proba[indices]),
)

# Conclusion

With Ginsu, we identified a subset of 33 passengers on which the model performs significantly worse. Those passengers:
- were in 3rd class (`num__pclass=3`),
- were between 37 and 39 years old (`num__age='[36.90, 39.20)'`),
- without any parents or children aboard (`num__parch=0.0`),
- and embarked in Queenstown (`cat__embarked_Q=1`).

To improve the modelisation, we should focus on reducing the error on those passengers.